# SSD — Single Shot MultiBox Detector (2015)

_Papers · Computer Vision_

**Tile fixed default boxes over feature maps at several scales, and in one pass predict a class and a box offset for each.**

---

This notebook is a practice scaffold for the **SSD — Single Shot MultiBox Detector (2015)** lesson. We build it up one small step at a time: read the note above each cell, run the cell, watch what changes, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build SSD's multi-scale matching idea in four steps: (1) work the default-box scale formula and one IoU by hand, (2) write a generator that tiles default boxes over a feature-map grid, (3) make a test set of small and large objects, then (4) measure matching recall and show that only the multi-scale box set covers both sizes.

### Step 1 — Work the scale formula and one IoU by hand

SSD spaces its default-box scales evenly from `s_min` to `s_max` (Eqn. 4). At each scale it also makes boxes of several aspect ratios `a_r`, where `w = s_k·√a_r` and `h = s_k/√a_r` (so the area stays `s_k²`), plus one extra `a_r=1` box of size `√(s_k·s_{k+1})`.

We compute those scales, list the aspect-ratio boxes at one level, and define the IoU (intersection-over-union) helpers we'll reuse — checking one worked match against an expected value.

In [ ]:
import torch
import math

torch.manual_seed(0)

# Default-box scales (Eqn 4): evenly spaced from s_min to s_max over m levels.
s_min, s_max, m = 0.2, 0.9, 4
scales = [round(s_min + (s_max - s_min) / (m - 1) * (k - 1), 4) for k in range(1, m + 1)]
print("scales s_k (k=1..4):", scales)          # [0.2, 0.4333, 0.6667, 0.9]

# Aspect-ratio boxes at level 2: w = s_k*sqrt(a), h = s_k/sqrt(a), so area = s_k^2.
sk = scales[1]                                   # level 2
for a in (1, 2, 3, 0.5, 1/3):
    w = sk * math.sqrt(a)
    h = sk / math.sqrt(a)
    print(f"  ar={a:.3f}:  w={w:.4f}  h={h:.4f}  area={w*h:.4f}  (= s_k^2 = {sk**2:.4f})")

s_extra = math.sqrt(scales[1] * scales[2])       # extra ar=1 box: sqrt(s_k * s_{k+1})
print(f"  extra ar=1 box side s'_k = sqrt(s2*s3) = {s_extra:.4f}")

# (cx,cy,w,h) -> (x1,y1,x2,y2)
def corners(b):
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], -1)

# IoU between every box in a:(N,4) and b:(M,4), both in (cx,cy,w,h).
def iou_matrix(a, b):
    a, b = corners(a)[:, None], corners(b)[None]
    ix1 = torch.max(a[..., 0], b[..., 0])
    iy1 = torch.max(a[..., 1], b[..., 1])
    ix2 = torch.min(a[..., 2], b[..., 2])
    iy2 = torch.min(a[..., 3], b[..., 3])
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    aa = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    ba = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (aa + ba - inter + 1e-9)

db = torch.tensor([[0.75, 0.75, sk, sk]])        # square ar=1 box at (0.75,0.75)
gt = torch.tensor([[0.75, 0.75, 0.5, 0.5]])      # ground truth 0.5x0.5
print("worked-example IoU (expect ~0.751):", round(iou_matrix(db, gt).item(), 4))

### Step 2 — Tile default boxes over a feature-map grid

SSD attaches default boxes to feature maps of different resolutions. A coarse grid yields a few large boxes (good for big objects); a fine grid yields many small boxes (good for small objects).

The generator places one square box of a given side at the center of every cell of a `grid × grid` map. We build a coarse set, a fine set, and their union — the multi-scale set that SSD actually uses.

In [ ]:
# Multi-scale default-box generator: one square box of size 'side' at each cell center.
def make_default_boxes(grid, side):
    """One square box of size 'side' at each cell center of a grid x grid feature map."""
    boxes = []
    for i in range(grid):
        for j in range(grid):
            cx = (j + 0.5) / grid       # cell center x
            cy = (i + 0.5) / grid       # cell center y
            boxes.append([cx, cy, side, side])
    return torch.tensor(boxes)

db_coarse = make_default_boxes(3, 0.55)          # few big boxes  -> large objects
db_fine = make_default_boxes(6, 0.22)            # many small boxes -> small objects
db_multi = torch.cat([db_coarse, db_fine])       # SSD: both scales at once

### Step 3 — Make a test set of small and large objects

To probe coverage we need ground-truth objects of two clearly different sizes. We sample 150 small objects (side `0.2`) and 150 large ones (side `0.55`) at random positions that keep each box fully inside the unit image.

This split lets us measure, separately, how well each default-box set covers small vs large objects.

In [ ]:
# A test set of objects: 150 small (0.2) + 150 large (0.55), random positions.
torch.manual_seed(1)

def gen(n, size):
    out = []
    for _ in range(n):
        cx = torch.rand(1).item() * (1 - size) + size / 2     # keep box inside the image
        cy = torch.rand(1).item() * (1 - size) + size / 2
        out.append([cx, cy, size, size])
    return torch.tensor(out)

small = gen(150, 0.20)
large = gen(150, 0.55)
all_obj = torch.cat([small, large])

### Step 4 — Measure matching recall across box sets

A default box can only train an object it actually matches (IoU > 0.5), so **recall** — the fraction of ground-truth objects with at least one matched box — is the geometric ceiling on what the detector can ever learn.

We compute recall for the coarse-only, fine-only, and multi-scale sets, split by object size. The result is the SSD effect: each single scale covers only one size, while the multi-scale union catches both.

In [ ]:
# Matching recall: fraction of ground-truth objects with a matched default box (IoU>0.5).
def recall(boxes, gts, thr=0.5):
    best_iou_per_gt = iou_matrix(boxes, gts).max(0).values   # best default box per ground truth
    return (best_iou_per_gt > thr).float().mean().item()

print("\nMatching recall (IoU>0.5), purely geometric:")
for name, dbset in [("coarse-only", db_coarse), ("fine-only", db_fine), ("multi-scale", db_multi)]:
    r_small = recall(dbset, small)
    r_large = recall(dbset, large)
    r_all = recall(dbset, all_obj)
    print(f"  {name:12s}  small={r_small:.3f}  large={r_large:.3f}  all={r_all:.3f}")
# coarse-only   small=0.000  large=0.487  all=0.243   <- misses ALL small objects
# fine-only     small=0.413  large=0.000  all=0.207   <- misses ALL large objects
# multi-scale   small=0.413  large=0.487  all=0.450   <- catches BOTH (the SSD effect)
# (Our small run, not the paper's reported number.)

## Visualize the data & results

_Do multi-scale default boxes match both small and large objects, where a single scale matches only one size?_

### Step 1 — Re-declare the box generator, IoU, and test objects

This panel re-imports torch and re-defines the default-box generator, the corner/IoU helpers, and the small/large test objects so it runs independently of the cells above. The logic matches the reference implementation; it is written compactly here.

In [ ]:
import torch

torch.manual_seed(0)

# A single default-box scale covers one object size; the multi-scale union covers both.
def make_default_boxes(grid, side):
    return torch.tensor([[(j + 0.5) / grid, (i + 0.5) / grid, side, side]
                         for i in range(grid) for j in range(grid)])

def corners(b):
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], -1)

def iou_matrix(a, b):
    a, b = corners(a)[:, None], corners(b)[None]
    ix1 = torch.max(a[..., 0], b[..., 0])
    iy1 = torch.max(a[..., 1], b[..., 1])
    ix2 = torch.min(a[..., 2], b[..., 2])
    iy2 = torch.min(a[..., 3], b[..., 3])
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    aa = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    ba = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (aa + ba - inter + 1e-9)

db_coarse = make_default_boxes(3, 0.55)
db_fine = make_default_boxes(6, 0.22)
db_multi = torch.cat([db_coarse, db_fine])

torch.manual_seed(1)

def gen(n, size):
    return torch.tensor([[torch.rand(1).item() * (1 - size) + size/2,
                          torch.rand(1).item() * (1 - size) + size/2, size, size]
                         for _ in range(n)])

small = gen(150, 0.20)
large = gen(150, 0.55)

### Step 2 — Compare recall per box set, split by object size

Now we compute the small-vs-large recall for each box set. The numbers behind the panel: coarse-only catches large objects but zero small ones, fine-only catches small but zero large, and the multi-scale union catches both — the qualitative SSD effect.

In [ ]:
def recall(boxes, gts, thr=0.5):
    return (iou_matrix(boxes, gts).max(0).values > thr).float().mean().item()

for name, db in [("coarse-only", db_coarse), ("fine-only", db_fine), ("multi-scale", db_multi)]:
    print(f"{name:12s}  small={recall(db, small):.3f}  large={recall(db, large):.3f}")
# coarse-only   small=0.000  large=0.487
# fine-only     small=0.413  large=0.000
# multi-scale   small=0.413  large=0.487   (catches both -> the SSD effect)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a multi-scale default-box set (coarse big boxes + fine small boxes)
            that matches both small and large objects. Replace it with coarse boxes only (a single
            large scale) and recompute matching coverage on a test set of small and large objects. What
            happens to each object size, and what does that demonstrate about why SSD is multi-scale?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep everything else identical &mdash; same test objects, same IoU>0.5 rule &mdash; and change only the default-box set from multi-scale to coarse-only. — _An honest ablation changes exactly one thing &mdash; the set of scales &mdash; so any difference in coverage is attributable to it._
- Measure the fraction of ground-truth objects that get at least one matched default box (recall), split by small vs large. — _A box can only train an object it actually matches; recall is the geometric ceiling on what the detector can ever learn._
- Observe that coarse-only boxes match large objects but score ~0 on small ones; the fine scale is what recovers the small objects. — _A big default box never reaches IoU>0.5 with a tiny ground-truth box, so small objects go unmatched without a small-scale map._

**Answer:** With coarse-only boxes, large objects still match (decent recall) but small
                 objects collapse to ~0 recall &mdash; no default box overlaps them enough. Adding back the
                 fine scale restores the small-object matches while leaving large ones untouched. Since only
                 the box-scale set changed, this isolates multi-scale default boxes as the reason SSD
                 handles objects of many sizes &mdash; the core contribution. The CODEVIZ panel shows exactly
                 this: coarse-only catches large, fine-only catches small, multi-scale catches both.

</details>

**Problem 2.** Using $m = 4$, $s_{\min} = 0.2$, $s_{\max} = 0.9$, compute the four default-box scales
            $s_1, \ldots, s_4$. Then give the width and height of the $a_r = 2$ box at level 3.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the step: $(s_{\max} - s_{\min})/(m-1) = (0.9 - 0.2)/3 = 0.2333$. — _Eqn 4 spaces the scales evenly; the step is the gap between consecutive levels._
- Add the step: $s_1 = 0.2$, $s_2 = 0.4333$, $s_3 = 0.6667$, $s_4 = 0.9$. — _$s_k = s_{\min} + \text{step}\cdot(k-1)$, so each level is one step larger._
- At level 3, $s_3 = 0.6667$, $a_r = 2$: $w = 0.6667\sqrt{2} = 0.9428$, $h = 0.6667/\sqrt{2} = 0.4714$. — _$w = s_k\sqrt{a_r}$, $h = s_k/\sqrt{a_r}$; area $= w h = 0.6667^2 = 0.4445$ is preserved._

**Answer:** $s_1 = 0.2,\ s_2 = 0.4333,\ s_3 = 0.6667,\ s_4 = 0.9$. The level-3 $a_r{=}2$ box is
                 $w = 0.9428$, $h = 0.4714$ (a wide box), with area $0.4445 = s_3^2$. These match the
                 notebook's first cell.

</details>

**Problem 3.** Your worked example matched a square default box of side $0.4333$ at center $(0.75,0.75)$ to a
            $0.5\times0.5$ ground truth at the same center, giving IoU $\approx 0.751$. Now shrink the ground
            truth to $0.15\times0.15$ (a small object) at $(0.75,0.75)$, keeping the same default box. Roughly
            what is the new IoU, and is it still a positive match?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The small ground truth ($0.15\times0.15$, area $0.0225$) sits entirely inside the big default box ($0.4333\times0.4333$, area $0.1877$), so intersection $=$ the small box's area $= 0.0225$. — _When one box is fully contained in the other, the intersection equals the smaller area._
- Union $=$ big area $+$ small area $-$ intersection $= 0.1877 + 0.0225 - 0.0225 = 0.1877$. — _The contained box adds no new area to the union beyond the big box._
- IoU $= 0.0225 / 0.1877 \approx 0.12$, which is well below $0.5$ &mdash; NOT a positive match. — _A big default box swallows a tiny object but overlaps it by only a small fraction of the union; IoU stays low._

**Answer:** IoU $\approx 0.0225/0.1877 \approx \mathbf{0.12}$ &mdash; far below the $0.5$ threshold, so
                 the big default box does not match the small object. This is precisely why SSD needs a
                 fine, small-scale feature map: only a small default box can reach IoU $\gt 0.5$ with a
                 small ground truth. One scale cannot cover both sizes &mdash; the multi-scale motivation in a
                 single calculation.

</details>